# Approach 16: Cross-domain test on PTB-XL

Run the Approach 12 fine-tuned model on PTB-XL (German cohort, ~0% Chagas prevalence) and check the false positive rate, split by RBBB / LAFB status.

In [ ]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

sns.set_context("notebook")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 5)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("chagas")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"PyTorch {torch.__version__} | Device : {device}")

/Users/jwasieleski/Prywatne/jul/workspace/magisterka/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch 2.10.0 | Device : mps


In [ ]:
CFG = {
    "ptbxl_cache": "preprocessed_cache.h5",
    "brazil_cache": "preprocessed_cache_brazil.h5",
    "ptbxl_dir": "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3",
    
    "model_path": "approach12_finetune_code15_best.pt",
    "ssl_model_path": "DeepECG_Docker-main/ssl_pretrained_model/SSL_pretrained.pt",
    "fairseq_path": "DeepECG_Docker-main/fairseq-signals",
    "deepecg_path": "DeepECG_Docker-main",
    
    "batch_size": 64,
    "val_fraction": 0.15,
    "test_fraction": 0.15,
    "random_seed": 42,
    "num_workers": 0,
}

In [ ]:
sys.path.append(os.path.abspath(CFG["deepecg_path"]))
sys.path.append(os.path.abspath(CFG["fairseq_path"]))

from fairseq_signals.utils import checkpoint_utils

class DeepECG_SSL_Classifier(nn.Module):
    def __init__(self, ssl_model_path):
        super().__init__()
        logger.info(f"Loading DeepECG-SSL from {ssl_model_path}")
        model, cfg, task = checkpoint_utils.load_model_and_task(
            ssl_model_path,
            suffix=""
        )
        self.encoder = model
        
        self.head = nn.Sequential(
            nn.Linear(768, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
        
    def forward(self, x):
        out = self.encoder(source=x, padding_mask=None, features_only=True)
        features = out["x"]
        if features.size(0) != x.size(0) and features.size(1) == x.size(0):
            features = features.transpose(0, 1)
        pooled = features.mean(dim=1)
        logits = self.head(pooled).view(-1)
        return logits

logger.info(f"Loading Fine-Tuned Model from {CFG['model_path']}")
model = DeepECG_SSL_Classifier(CFG["ssl_model_path"])
if os.path.exists(CFG["model_path"]):
    ckpt = torch.load(CFG["model_path"], map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        model.load_state_dict(ckpt["state_dict"])
        threshold_from_ckpt = ckpt.get("val_thr", None)
    else:
        model.load_state_dict(ckpt)
        threshold_from_ckpt = None
    logger.info(f"Checkpoint threshold: {threshold_from_ckpt}")
else:
    logger.warning(f"Model weights not found at {CFG['model_path']}. Using untrained head.")
    threshold_from_ckpt = None
model.to(device)
model.eval()

In [ ]:
cache_file = CFG["ptbxl_cache"]
assert Path(cache_file).exists(), f"Missing {cache_file}."

with h5py.File(cache_file, "r") as f:
    all_exam_ids = f["exam_ids"][:] if "exam_ids" in f else np.arange(f["labels"].shape[0])

ptbxl_csv = os.path.join(CFG["ptbxl_dir"], "ptbxl_database.csv")
if os.path.exists(ptbxl_csv):
    df_ptbxl = pd.read_csv(ptbxl_csv, index_col='ecg_id')
    import ast
    df_ptbxl.scp_codes = df_ptbxl.scp_codes.apply(lambda x: ast.literal_eval(x))
    
    def has_rbbb(codes):
        return any(c in ['CRBBB', 'IRBBB', 'RBBB'] for c in codes.keys())
    
    def has_lafb(codes):
        return any(c in ['LAFB', 'LPFB'] for c in codes.keys())
    
    df_ptbxl['has_rbbb'] = df_ptbxl.scp_codes.apply(has_rbbb)
    df_ptbxl['has_lafb'] = df_ptbxl.scp_codes.apply(has_lafb)
    
    ptbxl_ids = set(df_ptbxl.index.astype(str))
    
    ptbxl_indices = []
    for i, eid in enumerate(all_exam_ids):
        if str(eid) in ptbxl_ids:
            ptbxl_indices.append(i)
            
    ptbxl_indices = np.array(ptbxl_indices)
    logger.info(f"Found {len(ptbxl_indices)} PTB-XL exams in cache.")
else:
    logger.error(f"PTB-XL CSV not found at {ptbxl_csv}")
    ptbxl_indices = np.array([])

class CachedDataset(Dataset):
    def __init__(self, cache_path, indices):
        self.cache_path = cache_path
        self.indices = np.sort(indices)
        self._file = None

    def __len__(self):
        return len(self.indices)

    def _f(self):
        if self._file is None:
            self._file = h5py.File(self.cache_path, "r")
        return self._file

    def __getitem__(self, idx):
        f = self._f()
        ri = self.indices[idx]
        x = torch.from_numpy(f["signals"][ri]).float()
        eid = f["exam_ids"][ri] if "exam_ids" in f else ri
        return x, str(eid)

pin = device.type == "cuda"
ptbxl_loader = DataLoader(CachedDataset(cache_file, ptbxl_indices), batch_size=CFG["batch_size"], shuffle=False, pin_memory=pin)

18:14:47 [INFO] Found 2783 PTB-XL exams in cache.


In [ ]:
@torch.no_grad()
def run_inference(model, loader, device):
    all_preds = []
    all_eids = []
    
    for x, eids in tqdm(loader, desc="Inference"):
        x = x.to(device)
        logits = model(x)
        probs = torch.sigmoid(logits)
        all_preds.extend(probs.cpu().numpy())
        all_eids.extend(eids)
        
    return pd.DataFrame({"ecg_id": all_eids, "prob_chagas": all_preds})

if len(ptbxl_indices) > 0:
    df_preds = run_inference(model, ptbxl_loader, device)
    df_preds['ecg_id'] = df_preds['ecg_id'].astype(int)
    
    df_merged = pd.merge(df_preds, df_ptbxl.reset_index(), on="ecg_id", how="inner")
    
    if threshold_from_ckpt is not None:
        threshold = threshold_from_ckpt
        logger.info(f"Using threshold from checkpoint: {threshold:.4f}")
    else:
        logger.info("No threshold in checkpoint. Evaluating on Validation Set to find optimal threshold...")
        from sklearn.model_selection import GroupShuffleSplit
        from sklearn.metrics import roc_curve
        
        brazil_cache = CFG.get("brazil_cache", "preprocessed_cache_brazil.h5")
        if Path(brazil_cache).exists():
            with h5py.File(brazil_cache, "r") as f:
                n_total = f["labels"].shape[0]
                all_labels = f["labels"][:]
                all_exam_ids = f["exam_ids"][:] if "exam_ids" in f else np.arange(n_total)
                
            code15_dir = "4916206"
            samitrop_dir = "sami-trop"

            df_code15 = pd.read_csv(os.path.join(code15_dir, "exams.csv")) if os.path.exists(os.path.join(code15_dir, "exams.csv")) else pd.DataFrame()
            df_samitrop = pd.read_csv(os.path.join(samitrop_dir, "exams.csv")) if os.path.exists(os.path.join(samitrop_dir, "exams.csv")) else pd.DataFrame()

            if not df_code15.empty: df_code15["source"] = "code15"
            if not df_samitrop.empty: df_samitrop["source"] = "samitrop"

            df_meta = pd.concat([df_code15, df_samitrop], ignore_index=True)

            if not df_meta.empty and "exam_id" in df_meta.columns and "patient_id" in df_meta.columns:
                exam_to_patient = dict(zip(df_meta["exam_id"], df_meta["patient_id"]))
                exam_to_source = dict(zip(df_meta["exam_id"], df_meta["source"]))
                
                patient_ids = np.array([exam_to_patient.get(eid, eid) for eid in all_exam_ids])
                sources = np.array([exam_to_source.get(eid, "unknown") for eid in all_exam_ids])
                
                test_mask = (sources == "samitrop")
                train_val_indices = np.where(~test_mask)[0]
                
                gss = GroupShuffleSplit(n_splits=1, test_size=CFG["val_fraction"] / (1.0 - CFG["test_fraction"]), random_state=CFG["random_seed"])
                train_idx_rel, val_idx_rel = next(gss.split(train_val_indices, groups=patient_ids[train_val_indices]))
                
                val_indices = train_val_indices[val_idx_rel]
            else:
                logger.warning("Could not load metadata for GroupShuffleSplit. Using random split.")
                n_test = int(n_total * CFG["test_fraction"])
                n_val = int(n_total * CFG["val_fraction"])
                n_train = n_total - n_val - n_test
                rng = np.random.RandomState(CFG["random_seed"])
                perm = rng.permutation(n_total)
                val_indices = perm[n_train : n_train + n_val]
            
            class ValDataset(Dataset):
                def __init__(self, cache_path, indices):
                    self.cache_path = cache_path
                    self.indices = np.sort(indices)
                    self._file = None
                    with h5py.File(cache_path, "r") as f:
                        self.labels = f["labels"][self.indices]

                def __len__(self):
                    return len(self.indices)

                def _f(self):
                    if self._file is None:
                        self._file = h5py.File(self.cache_path, "r")
                    return self._file

                def __getitem__(self, idx):
                    f = self._f()
                    ri = self.indices[idx]
                    x = torch.from_numpy(f["signals"][ri]).float()
                    y = self.labels[idx]
                    return x, y

            val_loader = DataLoader(ValDataset(brazil_cache, val_indices), batch_size=CFG["batch_size"], shuffle=False)
            
            all_val_preds, all_val_targets = [], []
            for x, y in tqdm(val_loader, desc="Val Inference"):
                x = x.to(device)
                with torch.no_grad():
                    logits = model(x)
                probs = torch.sigmoid(logits)
                all_val_preds.extend(probs.cpu().numpy())
                all_val_targets.extend(y.numpy())
                
            fpr, tpr, thresholds = roc_curve(all_val_targets, all_val_preds)
            j_scores = tpr - fpr
            best_idx = np.argmax(j_scores)
            threshold = thresholds[best_idx]
            logger.info(f"Optimal Threshold (Youden's J) found: {threshold:.4f}")
        else:
            logger.warning(f"Could not find {brazil_cache}. Using default threshold 0.5")
            threshold = 0.5

    df_merged['pred_chagas'] = (df_merged['prob_chagas'] >= threshold).astype(int)
    
    fpr_overall = df_merged['pred_chagas'].mean()
    fpr_rbbb = df_merged[df_merged['has_rbbb']]['pred_chagas'].mean()
    fpr_lafb = df_merged[df_merged['has_lafb']]['pred_chagas'].mean()
    fpr_both = df_merged[df_merged['has_rbbb'] & df_merged['has_lafb']]['pred_chagas'].mean()
    fpr_neither = df_merged[~df_merged['has_rbbb'] & ~df_merged['has_lafb']]['pred_chagas'].mean()
    
    logger.info(f"Overall FPR on PTB-XL: {fpr_overall*100:.2f}%")
    logger.info(f"FPR on RBBB patients: {fpr_rbbb*100:.2f}%")
    logger.info(f"FPR on LAFB patients: {fpr_lafb*100:.2f}%")
    logger.info(f"FPR on RBBB+LAFB patients: {fpr_both*100:.2f}%")
    logger.info(f"FPR on patients with neither: {fpr_neither*100:.2f}%")
    
    plt.figure(figsize=(10, 6))
    sns.kdeplot(data=df_merged, x="prob_chagas", hue="has_rbbb", common_norm=False, fill=True)
    plt.title("Distribution of Predicted Chagas Probability on PTB-XL (by RBBB status)")
    plt.xlabel("Predicted Probability of Chagas")
    plt.axvline(threshold, color='r', linestyle='--', label=f'Threshold ({threshold})')
    plt.legend()
    plt.show()